In [25]:
import pandas as pd
import numpy as np

In [2]:
import yfinance as yf

In [3]:
import time
from datetime import date

In [4]:
from pathlib import Path

In [26]:
DATA_DIR = Path("data/raw")

files = list(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(files)}")

for file in files:
    print(file.name)

Number of CSV files: 6
canada_cpi_monthly.csv
canada_labour_market_monthly.csv
canada_bank_of_canada_daily.csv
canada_stocks_daily.csv
canada_gdp_monthly.csv
canada_bcpi_monthly.csv


In [27]:
for file in files:

    df = pd.read_csv(file)

    print("=" * 60)
    print(f"File: {file.name}")
    print(f"Shape: {df.shape}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

File: canada_cpi_monthly.csv
Shape: (1353, 3)

Columns:
['Date', 'CPI', 'Inflation_YoY']

Data Types:
Date              object
CPI              float64
Inflation_YoY    float64
dtype: object

Missing Values:
Date              0
CPI               0
Inflation_YoY    12
dtype: int64

Duplicate Rows:
0
File: canada_labour_market_monthly.csv
Shape: (610, 5)

Columns:
['Date', 'Employment', 'Labour force', 'Participation rate', 'Unemployment rate']

Data Types:
Date                   object
Employment            float64
Labour force          float64
Participation rate    float64
Unemployment rate     float64
dtype: object

Missing Values:
Date                  0
Employment            0
Labour force          0
Participation rate    0
Unemployment rate     0
dtype: int64

Duplicate Rows:
0
File: canada_bank_of_canada_daily.csv
Shape: (6650, 4)

Columns:
['Date', 'Policy_Rate', 'Bond_Yield_10Y', 'USD_CAD']

Data Types:
Date               object
Policy_Rate       float64
Bond_Yield_10Y    float6

In [28]:
for file in files:
    df = pd.read_csv(file)

    date_col = "Date" if "Date" in df.columns else "date"

    df[date_col] = pd.to_datetime(df[date_col])

    print("=" * 50)
    print(file.name)
    print("Start:", df[date_col].min())
    print("End:", df[date_col].max())
    print("Unique dates:", df[date_col].nunique())
    print("Duplicate dates:", df[date_col].duplicated().sum())

canada_cpi_monthly.csv
Start: 1914-01-01 00:00:00
End: 2026-09-01 00:00:00
Unique dates: 1353
Duplicate dates: 0
canada_labour_market_monthly.csv
Start: 1976-01-01 00:00:00
End: 2026-10-01 00:00:00
Unique dates: 610
Duplicate dates: 0
canada_bank_of_canada_daily.csv
Start: 2001-01-02 00:00:00
End: 2026-11-06 00:00:00
Unique dates: 6650
Duplicate dates: 0
canada_stocks_daily.csv
Start: 2000-01-04 00:00:00
End: 2026-11-06 00:00:00
Unique dates: 6745
Duplicate dates: 56830
canada_gdp_monthly.csv
Start: 1997-01-01 00:00:00
End: 2026-08-01 00:00:00
Unique dates: 356
Duplicate dates: 0
canada_bcpi_monthly.csv
Start: 1972-01-01 00:00:00
End: 2026-10-01 00:00:00
Unique dates: 658
Duplicate dates: 0


In [29]:
stocks = pd.read_csv(DATA_DIR / "canada_stocks_daily.csv")

stocks["Date"] = pd.to_datetime(stocks["Date"])

print("Total records:", len(stocks))

print("\nDuplicate Date-Ticker:")
print(stocks.duplicated(["Date", "Ticker"]).sum())

print("\nZero or Negative Prices:")
print((stocks["Close"] <= 0).sum())

print("\nInvalid OHLC:")
invalid_ohlc = (
    (stocks["High"] < stocks["Low"]) |
    (stocks["High"] < stocks["Open"]) |
    (stocks["High"] < stocks["Close"]) |
    (stocks["Low"] > stocks["Open"]) |
    (stocks["Low"] > stocks["Close"])
)

print(invalid_ohlc.sum())

print("\nZero Volume:")
print((stocks["Volume"] == 0).sum())

print("\nRecords per Ticker:")
print(stocks.groupby("Ticker").size())

print("\nInvalid Price Records:")
display(stocks[stocks["Close"] <= 0])

Total records: 63575

Duplicate Date-Ticker:
0

Zero or Negative Prices:
1

Invalid OHLC:
0

Zero Volume:
5

Records per Ticker:
Ticker
ABX.TO     6744
BMO.TO     6744
BN.TO      6744
CNQ.TO     6744
CNR.TO     6744
ENB.TO     6745
RY.TO      6744
SHOP.TO    2878
SU.TO      6744
TD.TO      6744
dtype: int64

Invalid Price Records:


,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits
34570,2003-05-19,ENB.TO,0.0,0.0,0.0,0.0,0.0,0,0.10375,0.0


In [30]:
monthly_files = [
    "canada_cpi_monthly.csv",
    "canada_gdp_monthly.csv",
    "canada_labour_market_monthly.csv",
    "canada_bcpi_monthly.csv"
]

for filename in monthly_files:

    df = pd.read_csv(DATA_DIR / filename)

    date_col = "Date" if "Date" in df.columns else "date"

    dates = pd.to_datetime(df[date_col])

    expected_months = pd.period_range(
        start=dates.min(),
        end=dates.max(),
        freq="M"
    )

    actual_months = dates.dt.to_period("M")

    missing_months = expected_months.difference(actual_months)

    print("=" * 50)
    print(filename)
    print("Expected months:", len(expected_months))
    print("Actual months:", len(actual_months))
    print("Missing months:", len(missing_months))
    print("Missing:", missing_months.tolist()[:15])

canada_cpi_monthly.csv
Expected months: 1353
Actual months: 1353
Missing months: 0
Missing: []
canada_gdp_monthly.csv
Expected months: 356
Actual months: 356
Missing months: 0
Missing: []
canada_labour_market_monthly.csv
Expected months: 610
Actual months: 610
Missing months: 0
Missing: []
canada_bcpi_monthly.csv
Expected months: 658
Actual months: 658
Missing months: 0
Missing: []


# Clean Data

In [31]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
stocks = pd.read_csv(RAW_DIR / "canada_stocks_daily.csv")
bank = pd.read_csv(RAW_DIR / "canada_bank_of_canada_daily.csv")
cpi = pd.read_csv(RAW_DIR / "canada_cpi_monthly.csv")
gdp = pd.read_csv(RAW_DIR / "canada_gdp_monthly.csv")
labour = pd.read_csv(RAW_DIR / "canada_labour_market_monthly.csv")
bcpi = pd.read_csv(RAW_DIR / "canada_bcpi_monthly.csv")

# Convert dates
for df in [stocks, bank, cpi, gdp, labour]:
    df["Date"] = pd.to_datetime(df["Date"])

bcpi["date"] = pd.to_datetime(bcpi["date"])
bcpi = bcpi.rename(columns={"date": "Date"})

# Remove invalid stock prices
invalid_prices = stocks["Close"] <= 0

print("Invalid stock records:", invalid_prices.sum())

stocks = stocks.loc[~invalid_prices].copy()

# Sort datasets
stocks = stocks.sort_values(["Ticker", "Date"]).reset_index(drop=True)

bank = bank.sort_values("Date").reset_index(drop=True)
cpi = cpi.sort_values("Date").reset_index(drop=True)
gdp = gdp.sort_values("Date").reset_index(drop=True)
labour = labour.sort_values("Date").reset_index(drop=True)
bcpi = bcpi.sort_values("Date").reset_index(drop=True)

# Save cleaned datasets
datasets = {
    "canada_stocks_cleaned.csv": stocks,
    "canada_bank_of_canada_cleaned.csv": bank,
    "canada_cpi_cleaned.csv": cpi,
    "canada_gdp_cleaned.csv": gdp,
    "canada_labour_market_cleaned.csv": labour,
    "canada_bcpi_cleaned.csv": bcpi
}

for filename, df in datasets.items():
    df.to_csv(PROCESSED_DIR / filename, index=False)
    print(filename, df.shape)


Invalid stock records: 1
canada_stocks_cleaned.csv (63574, 10)
canada_bank_of_canada_cleaned.csv (6650, 4)
canada_cpi_cleaned.csv (1353, 3)
canada_gdp_cleaned.csv (356, 4)
canada_labour_market_cleaned.csv (610, 5)
canada_bcpi_cleaned.csv (658, 8)


# Data Preparation – Summary

Collected and prepared six datasets from Yahoo Finance, Statistics Canada, and the Bank of Canada, covering Canadian stock market activity and macroeconomic indicators.
The preparation process included data type conversion, filtering, sorting, missing-value analysis, duplicate detection, invalid-record removal, and chronological validation.
Ten TSX-listed stocks were selected, alongside indicators covering CPI, inflation, GDP, labour market conditions, interest rates, bond yields, exchange rates, and commodity prices.
One invalid stock record was removed, while potentially legitimate zero-volume observations and extreme returns were retained for further investigation.
The final output consists of six cleaned datasets stored in data/processed/, with original source files preserved in data/raw/.
Result: Six prepared datasets containing daily and monthly Canadian financial and economic information, ready for the Data Integration phase.
